# FE BAR Step 4 - Fraud risk model training (Layer 3)

Trains a LightGBM fraud classifier on `workshop_guy_catalog.fe_bar_a0.txn_features_gold`, logs to MLflow, and registers `workshop_guy_catalog.fe_bar_a0.fraud_risk_model@champion` in Unity Catalog.

* Part A: pre-flight governance check (pii membership, high-amount rows, unmasked spend ratio). Stops if it fails.
* Part B: time-based split, LightGBM with class-imbalance weighting, test metrics, SHAP, seeded-cohort lift, action thresholds, UC registration.

In [0]:
import json, time
import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

CATALOG, SCHEMA = "workshop_guy_catalog", "fe_bar_a0"
FEATURE_TABLE = f"{CATALOG}.{SCHEMA}.txn_features_gold"
MODEL_NAME = f"{CATALOG}.{SCHEMA}.fraud_risk_model"
EXPERIMENT = "/Users/alex.owen@databricks.com/fe_bar_fraud_risk_model"
LABEL = "is_fraud"

# Safety flag: False = train and evaluate only. No new UC model version, @champion is not moved.
REGISTER_MODEL = False
CHAMPION_RUN_ID = "d89beaca8f81447780da60ed2ea7c811"  # run behind fraud_risk_model@champion (v1)

FEATURES = ["amount", "amt_to_monthly_spend", "amt_to_cust_avg_30d", "is_cross_border",
            "geo_distance_from_home", "is_new_device", "is_online", "risk_tier", "risk_segment",
            "mcc_category", "txn_count_1h", "txn_count_24h", "secs_since_prev_txn", "hour_of_day",
            "is_night", "tenure"]
CATEGORICAL = ["risk_tier", "risk_segment", "mcc_category"]
NUMERIC = [f for f in FEATURES if f not in CATEGORICAL]
# Explicitly excluded: is_fraud (label), txn_id, customer_id, merchant_id, device_id, txn_ts,
# home_country, merchant_country, channel (already encoded as is_online)

mlflow.set_registry_uri("databricks-uc")
mlflow.set_experiment(EXPERIMENT)
print("Experiment:", EXPERIMENT)
print("Model:", MODEL_NAME)
print("Features (%d):" % len(FEATURES), FEATURES)
print("REGISTER_MODEL:", REGISTER_MODEL, "| champion run:", CHAMPION_RUN_ID)

Experiment: /Users/alex.owen@databricks.com/fe_bar_fraud_risk_model
Model: workshop_guy_catalog.fe_bar_a0.fraud_risk_model
Features (16): ['amount', 'amt_to_monthly_spend', 'amt_to_cust_avg_30d', 'is_cross_border', 'geo_distance_from_home', 'is_new_device', 'is_online', 'risk_tier', 'risk_segment', 'mcc_category', 'txn_count_1h', 'txn_count_24h', 'secs_since_prev_txn', 'hour_of_day', 'is_night', 'tenure']
REGISTER_MODEL: False | champion run: d89beaca8f81447780da60ed2ea7c811


In [0]:
# Part A: stop before training if governance would silently filter/mask the high-amount fraud signal
pre = spark.sql(f"""
  SELECT current_user() AS current_user,
         is_account_group_member('pii') AS is_pii_member,
         (SELECT count(*) FROM {FEATURE_TABLE}) AS total_rows,
         (SELECT count_if(amount > 10000) FROM {FEATURE_TABLE}) AS amount_gt_10000_rows,
         (SELECT count(amt_to_monthly_spend) FROM {FEATURE_TABLE}) AS non_null_amt_to_monthly_spend
""").collect()[0].asDict()
for k, v in pre.items():
    print(f"{k:32s} {v}")

if not pre["is_pii_member"] or pre["amount_gt_10000_rows"] == 0:
    raise RuntimeError(
        "PRE-FLIGHT FAILED: not a pii member or amount > 10000 rows are hidden. "
        "Training would use filtered/masked data. Stopping - flag to the owner.")
print("\nPRE-FLIGHT PASSED: pii member, high-amount rows visible, spend ratio unmasked.")

current_user                     alex.owen@databricks.com
is_pii_member                    True
total_rows                       464632
amount_gt_10000_rows             751
non_null_amt_to_monthly_spend    431471

PRE-FLIGHT PASSED: pii member, high-amount rows visible, spend ratio unmasked.


In [0]:
pdf = (spark.table(FEATURE_TABLE)
       .select("txn_id", "txn_ts", LABEL, *FEATURES)
       .toPandas()
       .sort_values(["txn_ts", "txn_id"])
       .reset_index(drop=True))
pdf[LABEL] = pdf[LABEL].astype(int)

# Fixed category levels so training and scoring encode identically
CAT_LEVELS = {c: sorted(pdf[c].dropna().unique().tolist()) for c in CATEGORICAL}

def prep_features(df):
    X = df[FEATURES].copy()
    for c in CATEGORICAL:
        X[c] = pd.Categorical(X[c], categories=CAT_LEVELS[c])
    for c in NUMERIC:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype("float64")
    return X

# Time split: earliest ~80% of txn_ts for train, latest ~20% for test
cutoff = pdf["txn_ts"].quantile(0.80)
train = pdf[pdf["txn_ts"] < cutoff].reset_index(drop=True)
test = pdf[pdf["txn_ts"] >= cutoff].reset_index(drop=True)
X_train, y_train = prep_features(train), train[LABEL].values
X_test, y_test = prep_features(test), test[LABEL].values

split = pd.DataFrame({
    "split": ["train", "test", "all"],
    "rows": [len(train), len(test), len(pdf)],
    "fraud_rows": [int(y_train.sum()), int(y_test.sum()), int(pdf[LABEL].sum())],
    "fraud_prevalence": [y_train.mean(), y_test.mean(), pdf[LABEL].mean()],
    "min_txn_ts": [train.txn_ts.min(), test.txn_ts.min(), pdf.txn_ts.min()],
    "max_txn_ts": [train.txn_ts.max(), test.txn_ts.max(), pdf.txn_ts.max()],
})
print("Cutoff timestamp (test = txn_ts >= cutoff):", cutoff)
print(split.to_string(index=False))
print("\nCategory levels:", CAT_LEVELS)

Cutoff timestamp (test = txn_ts >= cutoff): 2026-09-21 15:20:18
split   rows  fraud_rows  fraud_prevalence          min_txn_ts          max_txn_ts
train 371705        8796          0.023664 2026-04-20 00:06:09 2026-09-21 15:20:14
 test  92927        1735          0.018671 2026-09-21 15:20:19 2026-10-16 23:55:43
  all 464632       10531          0.022665 2026-04-20 00:06:09 2026-10-16 23:55:43

Category levels: {'risk_tier': ['high', 'low', 'medium'], 'risk_segment': ['high', 'low', 'medium'], 'mcc_category': ['dining', 'entertainment', 'groceries', 'online_services', 'retail', 'travel']}


In [0]:
mlflow.lightgbm.autolog(log_models=False, log_input_examples=False, silent=True)

scale_pos_weight = float((y_train == 0).sum() / (y_train == 1).sum())
print(f"Class imbalance: scale_pos_weight = neg/pos = {(y_train == 0).sum()}/{(y_train == 1).sum()} = {scale_pos_weight:.4f}")

PARAMS = dict(n_estimators=300, learning_rate=0.05, num_leaves=15, max_depth=5,
              min_child_samples=50, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
              scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1)

run = mlflow.start_run(run_name=f"lgbm_fraud_{time.strftime('%Y%m%d_%H%M%S')}")
RUN_ID = run.info.run_id
mlflow.log_params({"split_cutoff_ts": str(cutoff), "train_rows": len(train), "test_rows": len(test),
                   "scale_pos_weight_used": round(scale_pos_weight, 4), "features": ",".join(FEATURES),
                   "categorical_features": ",".join(CATEGORICAL), "split_method": "time_based_80_20"})

clf = lgb.LGBMClassifier(**PARAMS)
clf.fit(X_train, y_train, categorical_feature=CATEGORICAL)
booster = clf.booster_
test_score = clf.predict_proba(X_test)[:, 1]
print("Run ID:", RUN_ID, "| trees:", booster.num_trees())

Class imbalance: scale_pos_weight = neg/pos = 362909/8796 = 41.2584


2026/10/23 03:26:58 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


Run ID: 5c12ba7b458f4794a37a50aee6d8845e | trees: 300


In [0]:
order = np.argsort(-test_score, kind="stable")
def precision_at(k): return float(y_test[order[:k]].mean())
def recall_at(k): return float(y_test[order[:k]].sum() / y_test.sum())

metrics = {
    "test_roc_auc": float(roc_auc_score(y_test, test_score)),
    "test_pr_auc": float(average_precision_score(y_test, test_score)),
    "test_precision_at_100": precision_at(100),
    "test_precision_at_500": precision_at(500),
    "test_recall_at_500": recall_at(500),
    "test_fraud_prevalence": float(y_test.mean()),
    "overall_fraud_prevalence": float(pdf[LABEL].mean()),
}
mlflow.log_metrics(metrics)
print(pd.DataFrame({"metric": list(metrics), "value": [round(v, 4) for v in metrics.values()]}).to_string(index=False))
print(f"\nPR-AUC lift over prevalence baseline: {metrics['test_pr_auc'] / metrics['test_fraud_prevalence']:.1f}x")

                  metric  value
            test_roc_auc 0.8448
             test_pr_auc 0.2808
   test_precision_at_100 0.7800
   test_precision_at_500 0.4080
      test_recall_at_500 0.1176
   test_fraud_prevalence 0.0187
overall_fraud_prevalence 0.0227

PR-AUC lift over prevalence baseline: 15.0x


In [0]:
import shap, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# TreeSHAP via LightGBM pred_contrib on a 20K random sample of the test set (last column = base value)
shap_idx = np.random.RandomState(42).choice(len(X_test), size=min(20000, len(X_test)), replace=False)
X_shap = X_test.iloc[shap_idx]
contrib = booster.predict(X_shap, pred_contrib=True)
shap_vals = contrib[:, :-1]

mean_abs = (pd.DataFrame({"feature": FEATURES, "mean_abs_shap": np.abs(shap_vals).mean(axis=0)})
            .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
print("Mean absolute SHAP (log-odds), test sample n=%d" % len(X_shap))
print(mean_abs.to_string(index=False, float_format=lambda v: f"{v:.4f}"))

X_plot = X_shap.copy()
for c in CATEGORICAL:
    X_plot[c] = X_plot[c].cat.codes.replace(-1, np.nan)
plt.figure()
shap.summary_plot(shap_vals, X_plot, feature_names=FEATURES, show=False, max_display=16)
plt.tight_layout()
plt.savefig("/tmp/shap_summary.png", dpi=120, bbox_inches="tight")
plt.close("all")
mlflow.log_artifact("/tmp/shap_summary.png", artifact_path="shap")
mean_abs.to_csv("/tmp/mean_abs_shap.csv", index=False)
mlflow.log_artifact("/tmp/mean_abs_shap.csv", artifact_path="shap")
print("\nLogged artifacts: shap/shap_summary.png, shap/mean_abs_shap.csv")

Mean absolute SHAP (log-odds), test sample n=20000
               feature  mean_abs_shap
geo_distance_from_home         0.2560
             risk_tier         0.1859
          risk_segment         0.1721
  amt_to_monthly_spend         0.1675
                amount         0.1342
         is_new_device         0.1335
       is_cross_border         0.1011
             is_online         0.0925
          mcc_category         0.0516
   secs_since_prev_txn         0.0423
   amt_to_cust_avg_30d         0.0417
                tenure         0.0394
           hour_of_day         0.0269
         txn_count_24h         0.0053
          txn_count_1h         0.0007
              is_night         0.0000

Logged artifacts: shap/shap_summary.png, shap/mean_abs_shap.csv


In [0]:
# Seeded cohort: amt_to_monthly_spend in the top decile (p90 from the TRAIN window) AND cross-border AND new device
cohort_p90 = float(train["amt_to_monthly_spend"].quantile(0.90))
def in_cohort(df):
    return ((df["amt_to_monthly_spend"] >= cohort_p90) & (df["is_cross_border"] == 1) & (df["is_new_device"] == 1)).values

c = in_cohort(test)
top500 = np.zeros(len(test), dtype=bool); top500[order[:500]] = True
cohort_metrics = {
    "cohort_rows_test": int(c.sum()),
    "cohort_fraud_rate": float(y_test[c].mean()),
    "overall_fraud_rate": float(y_test.mean()),
    "cohort_fraud_lift": float(y_test[c].mean() / y_test.mean()),
    "mean_risk_score_cohort": float(test_score[c].mean()),
    "mean_risk_score_non_cohort": float(test_score[~c].mean()),
    "top500_share_in_cohort": float(c[top500].mean()),
    "pr_auc_non_cohort": float(average_precision_score(y_test[~c], test_score[~c])),
    "fraud_prevalence_non_cohort": float(y_test[~c].mean()),
}
mlflow.log_param("cohort_amt_ratio_p90", cohort_p90)
mlflow.log_metrics(cohort_metrics)
print(f"Cohort amt_to_monthly_spend p90 threshold (train): {cohort_p90:.4f}\n")
print(pd.DataFrame({"measure": list(cohort_metrics), "value": [round(v, 4) for v in cohort_metrics.values()]}).to_string(index=False))
print(f"\nNon-cohort PR-AUC {cohort_metrics['pr_auc_non_cohort']:.4f} vs non-cohort prevalence "
      f"{cohort_metrics['fraud_prevalence_non_cohort']:.4f} -> the model learned signal beyond the seeded rule.")

Cohort amt_to_monthly_spend p90 threshold (train): 0.5442

                    measure    value
           cohort_rows_test 116.0000
          cohort_fraud_rate   0.1810
         overall_fraud_rate   0.0187
          cohort_fraud_lift   9.6962
     mean_risk_score_cohort   0.8756
 mean_risk_score_non_cohort   0.2145
     top500_share_in_cohort   0.0100
          pr_auc_non_cohort   0.2809
fraud_prevalence_non_cohort   0.0185

Non-cohort PR-AUC 0.2809 vs non-cohort prevalence 0.0185 -> the model learned signal beyond the seeded rule.


In [0]:
# Rules (from the test-set precision/recall curve):
#   block  = lowest score threshold where precision >= 0.50 (auto-decline only when at least half are fraud)
#   review = highest score threshold that still reaches recall >= 0.60 (analyst queue catches most fraud)
#   approve = below review
BLOCK_MIN_PRECISION, REVIEW_MIN_RECALL = 0.50, 0.60
prec, rec, thr = precision_recall_curve(y_test, test_score)
prec, rec = prec[:-1], rec[:-1]
block_threshold = float(thr[np.where(prec >= BLOCK_MIN_PRECISION)[0][0]])
review_threshold = float(thr[np.where(rec >= REVIEW_MIN_RECALL)[0][-1]])
assert review_threshold < block_threshold

def action(score):
    return np.where(score >= block_threshold, "block", np.where(score >= review_threshold, "review", "approve"))

act = action(test_score)
bands = []
for a in ["block", "review", "approve"]:
    m = act == a
    bands.append({"band": a, "txns": int(m.sum()), "pct_volume": m.mean(), "fraud_in_band": int(y_test[m].sum()),
                  "precision": y_test[m].mean() if m.any() else np.nan,
                  "recall": y_test[m].sum() / y_test.sum()})
bands = pd.DataFrame(bands)
flagged = act != "approve"
print(f"block_threshold  = {block_threshold:.6f}  (lowest score with precision >= {BLOCK_MIN_PRECISION})")
print(f"review_threshold = {review_threshold:.6f}  (highest score with cumulative recall >= {REVIEW_MIN_RECALL})\n")
print(bands.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
print(f"\nblock+review combined: precision {y_test[flagged].mean():.4f}, recall {y_test[flagged].sum()/y_test.sum():.4f}, "
      f"{flagged.sum()} txns ({flagged.mean():.2%} of test volume)")
print("Note: scores are uncalibrated (scale_pos_weight inflates them), so thresholds sit high in [0,1].")

mlflow.log_params({"block_threshold": block_threshold, "review_threshold": review_threshold,
                   "block_rule": f"min precision {BLOCK_MIN_PRECISION}", "review_rule": f"min recall {REVIEW_MIN_RECALL}"})
bands.to_csv("/tmp/threshold_bands.csv", index=False)
mlflow.log_artifact("/tmp/threshold_bands.csv", artifact_path="thresholds")

block_threshold  = 0.950267  (lowest score with precision >= 0.5)
review_threshold = 0.871416  (highest score with cumulative recall >= 0.6)

   band  txns  pct_volume  fraud_in_band  precision  recall
  block   288      0.0031            144     0.5000  0.0830
 review  3567      0.0384            897     0.2515  0.5170
approve 89072      0.9585            694     0.0078  0.4000

block+review combined: precision 0.2700, recall 0.6000, 3855 txns (4.15% of test volume)
Note: scores are uncalibrated (scale_pos_weight inflates them), so thresholds sit high in [0,1].


In [0]:
class FraudRiskModel(mlflow.pyfunc.PythonModel):
    """LightGBM booster + fixed category levels. predict() returns risk_score; contributions() returns TreeSHAP."""
    def __init__(self, features, categorical, cat_levels):
        self.features, self.categorical, self.cat_levels = features, categorical, cat_levels
        self.booster = None
    def load_context(self, context):
        import lightgbm as lgb
        self.booster = lgb.Booster(model_file=context.artifacts["booster"])
    def prep(self, df):
        import pandas as pd
        X = df[self.features].copy()
        for c in self.features:
            if c in self.categorical:
                X[c] = pd.Categorical(X[c].astype("object"), categories=self.cat_levels[c])
            else:
                X[c] = pd.to_numeric(X[c], errors="coerce").astype("float64")
        return X
    def predict(self, context, model_input, params=None):
        return self.booster.predict(self.prep(model_input))
    def contributions(self, df):
        return self.booster.predict(self.prep(df), pred_contrib=True)

def to_model_input(df):
    X = df[FEATURES].copy()
    for c in CATEGORICAL:
        X[c] = X[c].astype("object").where(X[c].notna(), None)
    for c in NUMERIC:
        X[c] = X[c].astype("float64")
    return X

signature = ModelSignature(
    inputs=Schema([ColSpec("string" if f in CATEGORICAL else "double", f, required=False) for f in FEATURES]),
    outputs=Schema([ColSpec("double", "risk_score")]))
print("Signature:\n", signature)
client = MlflowClient(registry_uri="databricks-uc")

if REGISTER_MODEL:
    booster.save_model("/tmp/fraud_booster.txt")
    input_example = to_model_input(test.head(5))
    model_info = mlflow.pyfunc.log_model(
        name="model",
        python_model=FraudRiskModel(FEATURES, CATEGORICAL, CAT_LEVELS),
        artifacts={"booster": "/tmp/fraud_booster.txt"},
        signature=signature,
        input_example=input_example,
        pip_requirements=["lightgbm==4.7.0", f"pandas=={pd.__version__}", f"numpy=={np.__version__}"],
        registered_model_name=MODEL_NAME,
    )
    version = str(model_info.registered_model_version)
    client.set_registered_model_alias(MODEL_NAME, "champion", version)
    mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
    print(f"\nRegistered {MODEL_NAME} version {mv.version}, alias 'champion' -> version {mv.version}, run_id {mv.run_id}")
else:
    mv = client.get_model_version_by_alias(MODEL_NAME, "champion")
    version = str(mv.version)
    print(f"\nREGISTER_MODEL=False: skipped log_model/registration. No new version created, @champion not moved.")
    print(f"Current {MODEL_NAME}@champion -> version {mv.version}, run_id {mv.run_id}")
    assert mv.run_id == CHAMPION_RUN_ID, "champion run_id changed unexpectedly"

/databricks/python/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


Signature:
 inputs: 
  ['amount': double (optional), 'amt_to_monthly_spend': double (optional), 'amt_to_cust_avg_30d': double (optional), 'is_cross_border': double (optional), 'geo_distance_from_home': double (optional), 'is_new_device': double (optional), 'is_online': double (optional), 'risk_tier': string (optional), 'risk_segment': string (optional), 'mcc_category': string (optional), 'txn_count_1h': double (optional), 'txn_count_24h': double (optional), 'secs_since_prev_txn': double (optional), 'hour_of_day': double (optional), 'is_night': double (optional), 'tenure': double (optional)]
outputs: 
  ['risk_score': double (required)]
params: 
  None


REGISTER_MODEL=False: skipped log_model/registration. No new version created, @champion not moved.
Current workshop_guy_catalog.fe_bar_a0.fraud_risk_model@champion -> version 1, run_id d89beaca8f81447780da60ed2ea7c811


In [0]:
# Sanity check: compare the registered champion to the in-memory scores of this run
loaded = mlflow.pyfunc.load_model(f"models:/{MODEL_NAME}@champion")
chk = np.asarray(loaded.predict(to_model_input(test.head(2000))))
print("max |champion - this run in-memory| on 2000 test rows:", float(np.abs(chk - test_score[:2000]).max()))
mlflow.end_run()

# This run's metrics next to the champion run's metrics
champ_run = client.get_run(CHAMPION_RUN_ID)
this_run = client.get_run(RUN_ID)
rows = []
for k in sorted(metrics):
    rows.append({"metric": k, "champion_run": champ_run.data.metrics.get(k), "this_run": this_run.data.metrics.get(k, metrics[k])})
for k in ["block_threshold", "review_threshold", "cohort_amt_ratio_p90", "train_rows", "test_rows", "split_cutoff_ts"]:
    rows.append({"metric": k, "champion_run": champ_run.data.params.get(k), "this_run": this_run.data.params.get(k)})
cmp = pd.DataFrame(rows)
print(f"\nchampion run {CHAMPION_RUN_ID} vs this run {RUN_ID}")
print(cmp.to_string(index=False))

summary = {"run_id": RUN_ID, "register_model": REGISTER_MODEL, "model": MODEL_NAME, "champion_version": version, "alias": "champion",
           "block_threshold": block_threshold, "review_threshold": review_threshold,
           "cohort_amt_ratio_p90": cohort_p90, **{k: round(v, 4) for k, v in metrics.items()}}
print(json.dumps(summary, indent=2))

2026/10/23 03:28:33 WARNING mlflow.tracking.context.registry: Encountered unexpected error during resolving tags: getContext().extraContext() is not a CommandContext accessor, it is not available or is misspelled.


max |champion - this run in-memory| on 2000 test rows: 0.0

champion run d89beaca8f81447780da60ed2ea7c811 vs this run 5c12ba7b458f4794a37a50aee6d8845e
                  metric        champion_run            this_run
overall_fraud_prevalence            0.022665            0.022665
   test_fraud_prevalence            0.018671            0.018671
             test_pr_auc            0.280758            0.280758
   test_precision_at_100                0.78                0.78
   test_precision_at_500               0.408               0.408
      test_recall_at_500            0.117579            0.117579
            test_roc_auc            0.844812            0.844812
         block_threshold  0.9502674433961342  0.9502674433961342
        review_threshold  0.8714158290727556  0.8714158290727556
    cohort_amt_ratio_p90  0.5441955491030543  0.5441955491030543
              train_rows              371705              371705
               test_rows               92927               92927
    

In [0]:
%pip install -q lightgbm==4.7.0 shap==0.54.1
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
